In [ ]:
# ==========================================
# PySpark DataFrame Operations Practical
# ==========================================

# Install PySpark in Google Colab
!pip install pyspark

# Import required libraries
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql import functions as F

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("DataFrame Operations Practical") \
    .getOrCreate()

# -------------------------------
# a) Create an empty DataFrame
# -------------------------------
empty_df = spark.createDataFrame([], StructType([]))
print("Empty DataFrame:")
empty_df.show()

# -------------------------------
# b) Create an empty Dataset (In PySpark, Dataset = DataFrame with schema)
# -------------------------------
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True)
])
empty_dataset = spark.createDataFrame([], schema)
print("Empty Dataset (with schema):")
empty_dataset.show()

# -------------------------------
# Create sample DataFrame for rest of operations
# -------------------------------
data = [
    (1, "Alice", {"city": "Mumbai", "pincode": "400001"}, 50000),
    (2, "Bob", {"city": "Delhi", "pincode": "110001"}, 60000),
    (3, "Charlie", {"city": "Pune", "pincode": "411001"}, 55000)
]

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("address", StructType([
        StructField("city", StringType(), True),
        StructField("pincode", StringType(), True)
    ])),
    StructField("salary", IntegerType(), True)
])

df = spark.createDataFrame(data, schema)
print("Original DataFrame:")
df.show(truncate=False)

# -------------------------------
# c) Rename nested column
# -------------------------------
df_renamed = df.withColumnRenamed("address.city", "address_city")  # PySpark can't directly rename nested field
# Alternative: Use select with alias for nested
df_renamed = df.select(
    "id", "name",
    F.col("address.city").alias("city"),
    F.col("address.pincode").alias("pincode"),
    "salary"
)
print("After Renaming Nested Column:")
df_renamed.show()

# -------------------------------
# d) Adding or Updating a column
# -------------------------------
df_updated = df.withColumn("bonus", F.col("salary") * 0.10)
print("After Adding 'bonus' Column:")
df_updated.show()

# Update column
df_updated = df_updated.withColumn("bonus", F.col("bonus") + 1000)
print("After Updating 'bonus' Column:")
df_updated.show()

# -------------------------------
# e) Drop a column
# -------------------------------
df_dropped = df_updated.drop("bonus")
print("After Dropping 'bonus' Column:")
df_dropped.show()

# -------------------------------
# f) Adding literal constant to DataFrame
# -------------------------------
df_literal = df.withColumn("Country", F.lit("India"))
print("After Adding Literal Constant Column:")
df_literal.show()

# -------------------------------
# g) Changing column data type
# -------------------------------
df_casted = df.withColumn("salary", F.col("salary").cast("double"))
print("After Changing 'salary' Column Type to Double:")
df_casted.printSchema()
df_casted.show()

# -------------------------------
# h) Pivot and Unpivot a DataFrame
# -------------------------------
sales_data = [
    ("East", "Apple", 50),
    ("East", "Banana", 30),
    ("West", "Apple", 70),
    ("West", "Banana", 45)
]
sales_df = spark.createDataFrame(sales_data, ["region", "product", "amount"])

# Pivot
pivot_df = sales_df.groupBy("region").pivot("product").sum("amount")
print("Pivoted DataFrame:")
pivot_df.show()

# Unpivot (Melt equivalent in PySpark)
unpivot_df = pivot_df.select(
    "region",
    F.expr("stack(2, 'Apple', Apple, 'Banana', Banana) as (product, amount)")
)
print("Unpivoted DataFrame:")
unpivot_df.show()

# -------------------------------
# i) Create a DataFrame using StructType & StructField schema
# -------------------------------
schema2 = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("emp_name", StringType(), True),
    StructField("department", StringType(), True)
])

emp_data = [
    (101, "John", "IT"),
    (102, "Sara", "HR"),
    (103, "Mike", "Finance")
]

df_struct = spark.createDataFrame(emp_data, schema2)
print("DataFrame Created Using StructType & StructField Schema:")
df_struct.show()

# Stop Spark Session
spark.stop()
